<figure>
    <center> <img src="../../images/mai_banner.png"  style="width:1000px;height:180px;" ></center>
</figure>

# Laboratorio 07: Gradient Descent Vectorizado

En este laboratorio aplicaremos las estructuras vistas en el laboratorio anterior para implementar el descenso del gradiente vectorizado.

In [ ]:
# Importando código
%pip install numpy matplotlib --quiet

In [ ]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt

A continuación se resume parte de la notación que encontrarás, actualizada para trabajar con varias características.

| Notación general | Descripción | Python (si corresponde) |
|:---|:---|:---|
| $a$ | Escalar, sin negrita | |
| $\mathbf{a}$ | Vector, en negrita | |
| $\mathbf{A}$ | Matriz, letra mayúscula en negrita | |
| **Regresión** | | |
| $\mathbf{X}$ | Matriz de ejemplos de entrenamiento | `X_train` |
| $\mathbf{y}$ | Valores objetivo de los ejemplos de entrenamiento | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | Ejemplo de entrenamiento número $i$ | `X[i]`, `y[i]` |
| $m$ | Número de ejemplos de entrenamiento | `m` |
| $n$ | Número de características de cada ejemplo | `n` |
| $\mathbf{w}$ | Parámetro: peso | `w` |
| $b$ | Parámetro: sesgo o bias | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | Resultado de evaluar el modelo parametrizado por $\mathbf{w}$ y $b$ en $\mathbf{x}^{(i)}$: $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b$ | `f_wb` |

# Problema

Vamos a usar el ejemplo de predicción del precio de viviendas. El conjunto de datos de entrenamiento contiene tres ejemplos con cuatro características (tamaño, número de dormitorios, plantas y antigüedad), que se muestran en la siguiente tabla. Ten en cuenta que, a diferencia de los laboratorios anteriores, el tamaño está expresado en pies cuadrados (sqft) y no en miles de pies cuadrados. Esto causa un problema que resolveremos mas adelante.

| Tamaño (pies cuadrados) | Número de dormitorios | Número de plantas | Antigüedad de la vivienda | Precio (miles de dólares) |
| ----------------| ------------------- |----------------- |--------------|-------------- |  
| 2104            | 5                   | 1                | 45           | 460           |  
| 1416            | 3                   | 2                | 40           | 232           |  
| 852             | 2                   | 1                | 35           | 178           |  

Construirás un modelo de regresión lineal con estos valores para predecir el precio de otras viviendas. Por ejemplo, una casa de 1200 pies cuadrados, 3 dormitorios, 1 planta y 40 años de antigüedad.

Ejecuta la siguiente celda de código para crear las variables `X_train` y `y_train`.

In [ ]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

## Matriz X que contiene los ejemplos

Al igual que en la tabla anterior, los ejemplos se almacenan en una matriz de NumPy llamada `X_train`. Cada fila de la matriz representa un ejemplo. Si tienes $m$ ejemplos de entrenamiento ($m$ es tres en nuestro ejemplo) y hay $n$ características (cuatro en nuestro ejemplo), $\mathbf{X}$ es una matriz de dimensiones ($m$, $n$): $m$ filas y $n$ columnas.


$$\mathbf{X} = 
\begin{pmatrix}
 x^{(0)}_0 & x^{(0)}_1 & \cdots & x^{(0)}_{n-1} \\ 
 x^{(1)}_0 & x^{(1)}_1 & \cdots & x^{(1)}_{n-1} \\
 \cdots \\
 x^{(m-1)}_0 & x^{(m-1)}_1 & \cdots & x^{(m-1)}_{n-1} 
\end{pmatrix}
$$
Notación:
- $\mathbf{x}^{(i)}$ es el vector que contiene el ejemplo $i$: $\mathbf{x}^{(i)} = (x^{(i)}_0, x^{(i)}_1, \cdots, x^{(i)}_{n-1})$.
- $x^{(i)}_j$ es el elemento $j$ del ejemplo $i$. El superíndice entre paréntesis indica el número del ejemplo, mientras que el subíndice representa un elemento.

Muestra los datos de entrada.

In [ ]:
# data is stored in numpy array/matrix
print(f"X Shape: {X_train.shape}, X Type:{type(X_train)})")
print(X_train)
print(f"y Shape: {y_train.shape}, y Type:{type(y_train)})")
print(y_train)

## 2.2 Vector de parámetros $\mathbf{w}$ y $b$

* $\mathbf{w}$ es un vector con $n$ elementos.
  - Cada elemento contiene el parámetro asociado a una característica.
  - En nuestro conjunto de datos, $n$ es igual a 4.
  - Conceptualmente, lo representamos como un vector columna.

$$\mathbf{w} = \begin{pmatrix}
w_0 \\ 
w_1 \\
\cdots\\
w_{n-1}
\end{pmatrix}
$$
* $b$ es un parámetro escalar.

For demonstration, $\mathbf{w}$ and $b$ will be loaded with some initial selected values that are near the optimal. $\mathbf{w}$ is a 1-D NumPy vector.

In [ ]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

# 3. Predicción del modelo con múltiples variables

La predicción del modelo con múltiples variables se expresa mediante el siguiente modelo lineal:

$$
f_{\mathbf{w},b}(\mathbf{x}) = w_0x_0 + w_1x_1 + \cdots + w_{n-1}x_{n-1} + b
$$
(Ecuación 1)

O, en notación vectorial:
$$
f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b
$$
(Ecuación 2)

donde $\cdot$ representa el `producto punto` de dos vectores.

Para ilustrar el producto punto, implementaremos la predicción usando las ecuaciones (1) y (2).

## 3.1 Predicción individual elemento por elemento

En la predicción anterior, multiplicábamos el valor de una característica por un parámetro y sumábamos el parámetro de bias (**b**). Una extensión directa de esa implementación para trabajar con varias características consiste en implementar la ecuación (1) de arriba recorriendo cada elemento, multiplicándolo por su parámetro correspondiente y, al final, sumando el parámetro de bias.

In [ ]:
def predict_single_loop(x, w, b): 
    """
    single predict using linear regression
    
    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters    
      b (scalar):  model parameter     
      
    Returns:
      p (scalar):  prediction
    """
    n = x.shape[0]
    p = 0
    for i in range(n):
        p_i = x[i] * w[i]  
        p = p + p_i         
    p = p + b                
    return p

In [ ]:
# get a row from our training data
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# make a prediction
f_wb = predict_single_loop(x_vec, w_init, b_init)
print(f"f_wb shape {f_wb.shape}, prediction: {f_wb}")

Ponga atención en las dimensiones de `x_vec`. Es un vector 1-D de NumPy de 4 elements, (4,). El resultado, `f_wb` es un escalar que representa la predicción del precio.

## Implementación vectorial de la predicción

Observa que la ecuación (1) anterior puede implementarse mediante el producto punto, como se muestra en la ecuación (2). Podemos aprovechar las operaciones vectoriales para acelerar las predicciones.

Recuerda que, como vimos en el laboratorio 06, la función `np.dot()` de NumPy [[enlace](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)] permite calcular el producto punto de dos vectores.

In [ ]:
def predict(x, w, b): 
    """
    single predict using linear regression
    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters   
      b (scalar):             model parameter 
      
    Returns:
      p (scalar):  prediction
    """
    p = np.dot(x, w) + b     
    return p 

In [ ]:
# get a row from our training data
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# make a prediction
f_wb = predict(x_vec,w_init, b_init)
print(f"f_wb shape {f_wb.shape}, prediction: {f_wb}")

Los resultados y las dimensiones son los mismos que en la versión anterior, que utilizaba un ciclo. De ahora en adelante, usaremos `np.dot` para estas operaciones. La predicción ahora se reduce a una sola instrucción. La mayoría de las rutinas la implementarán directamente, en lugar de llamar a una rutina de predicción independiente.

# Cálculo del costo con múltiples variables

La ecuación de la función de costo con múltiples variables $J(\mathbf{w},b)$ es:

$$
J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2
$$
(Ecuación 3)

donde:

$$
f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b
$$
(Ecuación 4)

A diferencia de los laboratorios anteriores, $\mathbf{w}$ y $\mathbf{x}^{(i)}$ son vectores y no escalares, lo que permite manejar múltiples características.

A continuación se muestra una implementación de las ecuaciones (3) y (4). Ten en cuenta que utiliza un *patrón estándar de este curso*, en el que se emplea un ciclo `for` sobre los `m` ejemplos.

In [ ]:
def compute_cost(X, y, w, b): 
    """
    compute cost
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):                                
        f_wb_i = np.dot(X[i], w) + b           #(n,)(n,) = scalar (see np.dot)
        cost = cost + (f_wb_i - y[i])**2       #scalar
    cost = cost / (2 * m)                      #scalar    
    return cost

In [ ]:
# Compute and display cost using our pre-chosen optimal parameters. 
cost = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at optimal w : {cost}')

# Descenso del gradiente con multiple variables

$$
\begin{align*}
\text{repetir}&\text{ hasta la convergencia:} \; \lbrace \\
& w_j = w_j - \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \; & \text{para } j = 0, \ldots, n-1 \\
& b = b - \alpha \frac{\partial J(\mathbf{w},b)}{\partial b} \\
\rbrace
\end{align*}
$$
(Ecuación 5)

donde $n$ es el número de características, los parámetros $w_j$ y $b$ se actualizan simultáneamente, y donde:

$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)}
$$
(Ecuación 6)

$$
\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})
$$
(Ecuación 7)

* $m$ es el número de ejemplos de entrenamiento del conjunto de datos.
* $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ es la predicción del modelo, mientras que $y^{(i)}$ es el valor objetivo.

## Cálculo del gradiente con múltiples variables

A continuación se muestra una implementación de unicamente el gradiente basado en las ecuaciones (6) y (7). Existen muchas formas de implementarlo. En esta versión se utiliza:
- Un ciclo externo sobre los $m$ ejemplos.
    - $\frac{\partial J(\mathbf{w},b)}{\partial b}$ puede calcularse directamente para cada ejemplo y acumularse.
    - un segundo ciclo sobre las $n$ características:
        - se calcula $\frac{\partial J(\mathbf{w},b)}{\partial w_j}$ para cada $w_j$.

In [ ]:
def compute_gradient(X, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar):       The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape           # (number of examples, number of features)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]   
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]    
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m                                
        
    return dj_db, dj_dw

In [ ]:
#Compute and display gradient 
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
print(f'dj_db at initial w,b: {tmp_dj_db}')
print(f'dj_dw at initial w,b: \n {tmp_dj_dw}')

** Resultados Esperados **:   
dj_db at initial w,b: -1.6739251122999121e-06  
dj_dw at initial w,b:   
 [-2.73e-03 -6.27e-06 -2.22e-06 -6.92e-05]  

## Versión vectorizada del descenso del gradiente con multiples variables

La rutina que se presenta a continuación implementa la *ecuación 5* anteriormente mencionada.

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))    : target values
      w_in (ndarray (n,)) : initial model parameters  
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,)) : Updated values of parameters 
      b (scalar)       : Updated value of parameter 
      """
    
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_db,dj_dw = gradient_function(X, y, w, b)   

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            J_history.append( cost_function(X, y, w, b))

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")
        
    return w, b, J_history #return final w,b and J history for graphing

En la siguiente celda podremos a prueba la implementación.

In [ ]:
# initialize parameters
initial_w = np.zeros_like(w_init)
initial_b = 0.
# some gradient descent settings
iterations = 1000
alpha = 5.0e-7
# run gradient descent 
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient, 
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
m,_ = X_train.shape
for i in range(m):
    print(f"prediction: {np.dot(X_train[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

** Resultados Esperados **:    
b,w found by gradient descent: -0.00,[ 0.2   0.   -0.01 -0.07]   
prediction: 426.19, target value: 460  
prediction: 286.17, target value: 232  
prediction: 171.47, target value: 178  

In [ ]:
# plot cost versus iteration  
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist)
ax2.plot(100 + np.arange(len(J_hist[100:])), J_hist[100:])
ax1.set_title("Cost vs. iteration");  ax2.set_title("Cost vs. iteration (tail)")
ax1.set_ylabel('Cost')             ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')   ;  ax2.set_xlabel('iteration step') 
plt.show()

*¡Observa el resultado no es tan bueno!* El costo aún sigue disminuyendo y nuestras predicciones no son muy precisas. En el próximo laboratorio exploraremos cómo mejorar esto.

Continuara ...